# 23 — Week 6: ONE test run of the quantum models with the frozen protocol

Same protocol as notebook 16 (Week 4 benchmark): every choice was made on validation; the models are frozen
(checkpoints / fixed reservoirs with trained readouts, train only, no refit on train + validation); a reproduction cell
must re-create every stored validation forecast before the test split is opened; a freeze record (protocol, model list,
SHA-256 of all inputs and model files) is written first; the test split is opened once; any later fix is reported.

Frozen set (main configurations chosen in notebooks 18–19): hybrid QNN, its no-entanglement variant, MLP control and
bottleneck control (3 seeds each); QRC, its no-entanglement variant and the random-feature control (3 reservoirs each)
and the raw control; plus the hybrid QNN evaluated with the Heron r2-like noise of notebook 22 (inference only,
secondary). The Week 4 test forecasts (`reports/week4/test_predictions.parquet`) are reused for the comparison; they
are not recomputed. Data-fraction and qubit-sweep models stay validation-only.

**Run this notebook only after notebooks 18–22 are final.** If `reports/week6/quantum_test_freeze.json` already
exists, the test split was already opened once: cell 3 then writes a second, time-stamped record (report this).

In [1]:
# Cell 1: setup and the frozen model set
import sys, json, time, math, hashlib, subprocess, warnings
from pathlib import Path
import numpy as np
import polars as pl
import torch

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
from models import rnn as R
from models import quantum as Q
from models import qrc as QR
from evaluation import metrics as M
from evaluation import compare as C

HORIZONS = R.HORIZONS
REP5, REP4, REP = ROOT / "reports" / "week5", ROOT / "reports" / "week4", ROOT / "reports" / "week6"
REP.mkdir(parents=True, exist_ok=True)
qv = json.loads((REP5 / "qnn_val.json").read_text())
qr = json.loads((REP5 / "qrc_val.json").read_text())
MAIN, QNN_GROUP = qv["main_hybrid"], qv.get("group", "week5_qnn")
QRC_GROUP, QRC_N, QRC_DEPTH = "week5_qrc", int(qr["n_qubits"]), int(qr["depth"])
BEST_SCALE = {k: float(v) for k, v in qr["best_scale"].items()}
SEEDS = [0, 1, 2]
HYB = {"hqnn": Q.variant_name(MAIN), "hqnn_noent": Q.variant_name({**MAIN, "entangle": False}),
       "ctrl_mlp": Q.variant_name({**MAIN, "middle": "mlp"}), "ctrl_bottleneck": Q.variant_name({**MAIN, "middle": "bottleneck"})}
QRCM = {k: QR.variant_name(k, QRC_N, QRC_DEPTH) for k in ("qrc", "qrc_noent", "rf", "raw")}
HERON = {"p1": 3e-4, "p2": 3e-3, "p_ro": 1.25e-2, "shots": 1024}
NOISY = HYB["hqnn"] + "_heron_r2"
FREEZE_F = REP / "quantum_test_freeze.json"
DATA_CFG = {"scaling": "local", "calendar": False, "covariates": False}
md = pl.read_parquet(ROOT / "data" / "processed" / "model_dataset.parquet")
meta = json.loads((ROOT / "data" / "processed" / "model_dataset_meta.json").read_text())
data = R.Data(md, DATA_CFG)
print("hybrid family:", HYB, "\nQRC family:", QRC_N, "qubits", QRCM, "| scales", {k: round(v, 3) for k, v in BEST_SCALE.items()})
print("usable origins:", {h: {s: len(data.idx[h][s]) for s in ("val", "test")} for h in HORIZONS})
print("freeze record exists:", FREEZE_F.exists(), "(if True, the test split was already opened once)")


def qrc_run_dir(k, s):
    base = QRCM[k]
    return ROOT / "experiments" / QRC_GROUP / (f"{base}_s0" if k == "raw" else f"{base}_{QR.scale_tag(BEST_SCALE[k])}_s{s}")


def collect(split, noisy=False, log=print):
    """{h: {model: {seed: forecasts on data.idx[h][split]}}} for the frozen set."""
    out = {h: {} for h in HORIZONS}
    ta = data.origins(split)
    t0 = time.time()
    for key, name in HYB.items():
        for s in SEEDS:
            model, _ = Q.load_trained(ROOT / "experiments" / QNN_GROUP / f"{name}_s{s}", data)
            pv = Q.predict(model, data, ta)
            for j, h in enumerate(HORIZONS):
                out[h].setdefault(name, {})[s] = pv[np.searchsorted(ta, data.idx[h][split]), j]
            if noisy and key == "hqnn":
                pn = Q.predict(Q.noisy_copy(model, seed=1000 + s, **HERON), data, ta, batch=2048)
                for j, h in enumerate(HORIZONS):
                    out[h].setdefault(NOISY, {})[s] = pn[np.searchsorted(ta, data.idx[h][split]), j]
    log(f"  hybrid family{' (+ noisy)' if noisy else ''}: {time.time() - t0:.0f} s")
    t0 = time.time()
    U = QR.minute_inputs(data, md, QRC_N)
    for k, name in QRCM.items():
        for s in ([0] if k == "raw" else SEEDS):
            Phi = QR.reservoir_cached(ROOT / "experiments" / "_cache_qrc", k, U, QRC_DEPTH, BEST_SCALE[k], s)
            fits = QR.load_readout(qrc_run_dir(k, s) / "readout.npz")
            pr = QR.predict_split(fits, Phi, data, split)
            for h in HORIZONS:
                out[h].setdefault(name, {})[s] = pr[h]
    log(f"  QRC family: {time.time() - t0:.0f} s")
    return out

hybrid family: {'hqnn': 'hqnn_noreup_q4_l2', 'hqnn_noent': 'hqnn_noent_noreup_q4_l2', 'ctrl_mlp': 'ctrl_mlp_q4_l2', 'ctrl_bottleneck': 'ctrl_bottleneck_q4_l2'} 
QRC family: 8 qubits {'qrc': 'qrc_q8_d4', 'qrc_noent': 'qrc_noent_q8_d4', 'rf': 'ctrl_rf_q8_f40', 'raw': 'ctrl_raw_q8'} | scales {'qrc': 0.098, 'qrc_noent': 0.196, 'rf': 0.098, 'raw': 1.571}
usable origins: {1: {'val': 42575, 'test': 42575}, 5: {'val': 42567, 'test': 42567}, 15: {'val': 42547, 'test': 42547}}
freeze record exists: False (if True, the test split was already opened once)


In [2]:
# Cell 2: REPRODUCE every stored validation forecast from the frozen artifacts (test split untouched)
VAL = collect("val")
rows = []
for h in HORIZONS:
    for name, by_seed in VAL[h].items():
        for s, v in by_seed.items():
            if name in HYB.values():
                f = ROOT / "experiments" / QNN_GROUP / f"{name}_s{s}" / "predictions_val.parquet"
            else:
                k = [kk for kk, nn in QRCM.items() if nn == name][0]
                f = qrc_run_dir(k, s) / "predictions_val.parquet"
            ref = pl.read_parquet(f).filter(pl.col("h") == h).sort("minute_epoch")["yhat"].to_numpy()
            d = float(np.max(np.abs(np.asarray(v) - ref))) if len(ref) == len(v) else float("inf")
            rows.append({"h": h, "model": name, "seed": s, "max_abs_diff_kW": d, "ok": d < 1e-3})
rep = pl.DataFrame(rows)
print(rep.group_by("model").agg(pl.len().alias("checks"), pl.col("max_abs_diff_kW").max(), pl.col("ok").all()).sort("model"))
REPRO_OK = bool(rep["ok"].all())
print("\nREPRODUCTION PASSED: the frozen artifacts re-create the validation forecasts" if REPRO_OK
      else "\nREPRODUCTION FAILED: do NOT run the next cells; send this output")

  hybrid family: 3 s
  QRC family: 258 s
shape: (8, 4)
┌─────────────────────────┬────────┬─────────────────┬──────┐
│ model                   ┆ checks ┆ max_abs_diff_kW ┆ ok   │
│ ---                     ┆ ---    ┆ ---             ┆ ---  │
│ str                     ┆ u32    ┆ f64             ┆ bool │
╞═════════════════════════╪════════╪═════════════════╪══════╡
│ ctrl_bottleneck_q4_l2   ┆ 9      ┆ 0.0             ┆ true │
│ ctrl_mlp_q4_l2          ┆ 9      ┆ 0.0             ┆ true │
│ ctrl_raw_q8             ┆ 3      ┆ 0.0             ┆ true │
│ ctrl_rf_q8_f40          ┆ 9      ┆ 0.0             ┆ true │
│ hqnn_noent_noreup_q4_l2 ┆ 9      ┆ 0.0             ┆ true │
│ hqnn_noreup_q4_l2       ┆ 9      ┆ 0.0             ┆ true │
│ qrc_noent_q8_d4         ┆ 9      ┆ 0.0             ┆ true │
│ qrc_q8_d4               ┆ 9      ┆ 0.0             ┆ true │
└─────────────────────────┴────────┴─────────────────┴──────┘

REPRODUCTION PASSED: the frozen artifacts re-create the validation forecasts

In [3]:
# Cell 3: freeze record (written before the test split is opened)
assert REPRO_OK, "cell 2 must pass first"
def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
try:
    git = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip() or None
except Exception:  # noqa: BLE001
    git = None
files = ["data/processed/model_dataset.parquet", "src/models/quantum.py", "src/models/qrc.py"]
files += [str((ROOT / "experiments" / QNN_GROUP / f"{n}_s{s}" / "model.pt").relative_to(ROOT)) for n in HYB.values() for s in SEEDS]
files += [str((qrc_run_dir(k, s) / "readout.npz").relative_to(ROOT)) for k in QRCM for s in ([0] if k == "raw" else SEEDS)]
freeze = {"frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"), "git_head": git,
          "protocol": ["model set frozen (main configurations chosen on validation in notebooks 18-19)",
                       "train only, no refit on train+val", "hybrid models from saved checkpoints",
                       "QRC: fixed reservoirs (seeded), saved ridge readouts", "noise: inference only, Heron r2-like, 1024 shots",
                       "one test run; later fixes reported"],
          "main_hybrid": MAIN, "qrc": {"n_qubits": QRC_N, "depth": QRC_DEPTH, "scales": BEST_SCALE}, "noise": HERON,
          "models": sorted(set(HYB.values()) | set(QRCM.values()) | {NOISY}),
          "inputs_sha256": {f: sha(ROOT / f) for f in files},
          "val_reproduction_max_diff_kW": float(rep["max_abs_diff_kW"].max())}
if FREEZE_F.exists():
    print("NOTE: a freeze record already exists (the test split was opened before); a second record is written next to it")
    FREEZE_F = REP / f"quantum_test_freeze_{time.strftime('%Y%m%d_%H%M%S')}.json"
FREEZE_F.write_text(json.dumps(freeze, indent=1))
print(json.dumps({k: v for k, v in freeze.items() if k != "inputs_sha256"}, indent=1))
print(f"SHA-256 recorded for {len(files)} files")

{
 "frozen_at": "2026-10-12 13:48:34",
 "git_head": "96597605437198fbfb9cb0a5da4aee2b50ccbbd4",
 "protocol": [
  "model set frozen (main configurations chosen on validation in notebooks 18-19)",
  "train only, no refit on train+val",
  "hybrid models from saved checkpoints",
  "QRC: fixed reservoirs (seeded), saved ridge readouts",
  "noise: inference only, Heron r2-like, 1024 shots",
  "one test run; later fixes reported"
 ],
 "main_hybrid": {
  "n_qubits": 4,
  "n_layers": 2,
  "reupload": false,
  "entangle": true
 },
 "qrc": {
  "n_qubits": 8,
  "depth": 4,
  "scales": {
   "qrc": 0.09817477042468103,
   "qrc_noent": 0.19634954084936207,
   "rf": 0.09817477042468103,
   "raw": 1.5707963267948966
  }
 },
 "noise": {
  "p1": 0.0003,
  "p2": 0.003,
  "p_ro": 0.0125,
  "shots": 1024
 },
 "models": [
  "ctrl_bottleneck_q4_l2",
  "ctrl_mlp_q4_l2",
  "ctrl_raw_q8",
  "ctrl_rf_q8_f40",
  "hqnn_noent_noreup_q4_l2",
  "hqnn_noreup_q4_l2",
  "hqnn_noreup_q4_l2_heron_r2",
  "qrc_noent_q8_d4",


In [4]:
# Cell 4: OPEN THE TEST SPLIT: forecasts of the frozen quantum set (one run; the noisy hybrid takes ~10 min)
assert REPRO_OK and FREEZE_F.exists(), "cells 2 and 3 first"
TEST = collect("test", noisy=True)
parts = []
for h in HORIZONS:
    t = data.idx[h]["test"]
    for m, by_seed in TEST[h].items():
        for s, v in by_seed.items():
            parts.append(pl.DataFrame({"model": [m] * len(t), "seed": np.full(len(t), s, np.int16),
                                       "h": np.full(len(t), h, np.int16), "minute_epoch": md["minute_epoch"].to_numpy()[t],
                                       "y": data.P[t + h], "p0": data.P[t], "yhat": np.asarray(v, float)}))
PRED_TEST = pl.concat(parts)
PRED_TEST.write_parquet(REP / "quantum_test_predictions.parquet")
TEST_SHA = hashlib.sha256((REP / "quantum_test_predictions.parquet").read_bytes()).hexdigest()
fr = json.loads(FREEZE_F.read_text())
fr["test_predictions_sha256"] = TEST_SHA
fr["test_opened_at"] = time.strftime("%Y-%m-%d %H:%M:%S")
FREEZE_F.write_text(json.dumps(fr, indent=1))
print(f"test predictions: {PRED_TEST.height:,} rows | models {PRED_TEST['model'].n_unique()} | sha256 {TEST_SHA[:16]}...")

  hybrid family (+ noisy): 83 s
  QRC family: 15 s
test predictions: 3,192,225 rows | models 9 | sha256 334776b2b896728e...


In [5]:
# Cell 5: test metrics (mean +- sd over seeds), merged with the frozen Week 4 test benchmark; validation -> test
L24 = md["L24_kW"].to_numpy().astype(float)
MOD5 = md["mod5"].to_numpy().astype(int)
rows = []
for h in HORIZONS:
    t = data.idx[h]["test"]
    y, p0 = data.P[t + h], data.P[t]
    ramp = md[f"ramp_h{h}"].to_numpy().astype(float)[t]
    for m, by_seed in TEST[h].items():
        for s, v in by_seed.items():
            r = M.evaluate(y, v, p0, l24=L24[t], ramp=ramp, thr=meta["thresholds"][str(h)]["rel_p95"],
                           mod5_target=MOD5[t + h] if h == 1 else None, dm_lag=60)
            rows.append({"h": h, "model": m, "seed": s, **{k: r.get(k) for k in
                         ["n", "MAE", "RMSE", "sMAPE", "R2", "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal",
                          "ramp_recall", "ramp_F1", "MAE_no_mod0_target", "DM_p"]}})
RUNS = pl.DataFrame(rows)
RUNS.write_csv(REP / "quantum_test_runs.csv")
fam = lambda m: "hybrid quantum" if m.startswith("hqnn") else ("quantum reservoir" if m.startswith("qrc") else "classical control")
BT = (RUNS.group_by("h", "model").agg(pl.len().alias("n_seeds"), pl.col("MAE").mean().alias("MAE_mean"), pl.col("MAE").std().alias("MAE_sd"),
        *[pl.col(c).mean() for c in ["RMSE", "sMAPE", "R2", "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal", "ramp_recall",
                                     "ramp_F1", "MAE_no_mod0_target"]], pl.col("DM_p").max().alias("DM_p_max"))
        .with_columns(pl.col("model").map_elements(fam, return_dtype=pl.String).alias("family")))
W4 = pl.read_csv(REP4 / "benchmark_test.csv")
ALL = pl.concat([W4, BT], how="diagonal_relaxed").sort("h", "MAE_mean").with_columns(
        pl.col("MAE_mean").rank("ordinal").over("h").alias("rank"))
ALL.write_csv(REP / "benchmark_test_all.csv")
pl.Config.set_tbl_rows(80); pl.Config.set_tbl_width_chars(220)
SHOW = list(HYB.values()) + list(QRCM.values()) + [NOISY, "lightgbm", "lightgbm_cov", "patchtst", "gru_uni", "timesfm_2p5_200m", "persistence"]
for h in HORIZONS:
    print(f"\n=== TEST h = {h} min ({ALL.filter(pl.col('h') == h).height} models) ===")
    print(ALL.filter((pl.col("h") == h) & pl.col("model").is_in(SHOW)).select("rank", "family", "model", "n_seeds",
          pl.col("MAE_mean").round(4), pl.col("MAE_sd").round(4), pl.col("skill_MAE").round(3), pl.col("MAE_ramp").round(3),
          pl.col("ramp_F1").round(3)))
VALS = pl.concat([pl.read_csv(REP5 / "qnn_val_summary.csv").select("h", "model", pl.col("skill_MAE").alias("val_skill")),
                  pl.read_csv(REP5 / "qrc_val_summary.csv").select("h", "model", pl.col("skill_MAE").alias("val_skill"))])
cmp_ = BT.select("h", "model", pl.col("skill_MAE").alias("test_skill")).join(VALS, on=["h", "model"], how="left").sort("h", "model")
print("\nvalidation -> test skill of the Week 5 models:")
print(cmp_.with_columns(pl.col("test_skill").round(3), pl.col("val_skill").round(3)))
cmp_.write_csv(REP / "quantum_val_vs_test.csv")


=== TEST h = 1 min (29 models) ===
shape: (15, 9)
┌──────┬───────────────────┬────────────────────────────┬─────────┬───┬────────┬───────────┬──────────┬─────────┐
│ rank ┆ family            ┆ model                      ┆ n_seeds ┆ … ┆ MAE_sd ┆ skill_MAE ┆ MAE_ramp ┆ ramp_F1 │
│ ---  ┆ ---               ┆ ---                        ┆ ---     ┆   ┆ ---    ┆ ---       ┆ ---      ┆ ---     │
│ u32  ┆ str               ┆ str                        ┆ i64     ┆   ┆ f64    ┆ f64       ┆ f64      ┆ f64     │
╞══════╪═══════════════════╪════════════════════════════╪═════════╪═══╪════════╪═══════════╪══════════╪═════════╡
│ 1    ┆ ML                ┆ lightgbm_cov               ┆ 3       ┆ … ┆ 0.0001 ┆ 0.114     ┆ 1.005    ┆ 0.237   │
│ 2    ┆ ML                ┆ lightgbm                   ┆ 3       ┆ … ┆ 0.0001 ┆ 0.064     ┆ 1.118    ┆ 0.138   │
│ 3    ┆ foundation        ┆ timesfm_2p5_200m           ┆ 1       ┆ … ┆ null   ┆ 0.056     ┆ 1.125    ┆ 0.125   │
│ 6    ┆ deep              ┆ patchtst

In [6]:
# Cell 6: Diebold-Mariano with Holm (seed 0) on test: hybrid and QRC vs their controls and the classical references;
# ramp breakdown; weekly skill
W4P = pl.read_parquet(REP4 / "test_predictions.parquet").filter(pl.col("seed") == 0).drop("seed")
REFS = [m for m in ["lightgbm", "patchtst", "gru_uni", "persistence"] if m in W4P["model"].unique().to_list()]
NEW0 = PRED_TEST.filter(pl.col("seed") == 0).drop("seed")
pred = pl.concat([C._std(W4P.filter(pl.col("model").is_in(REFS))), C._std(NEW0)])
al, cov = C.align(pred)
al = C.add_labels(al, md)
DM_MODELS = [HYB["hqnn"], HYB["ctrl_mlp"], HYB["hqnn_noent"], HYB["ctrl_bottleneck"], QRCM["qrc"], QRCM["rf"], QRCM["raw"]] + REFS
dm = C.dm_pairs(al, {h: DM_MODELS for h in HORIZONS}, lag=60)
dm.write_csv(REP / "quantum_dm_pairs_test.csv")
key = dm.filter(pl.col("model_a").is_in([HYB["hqnn"], QRCM["qrc"]]) | pl.col("model_b").is_in([HYB["hqnn"], QRCM["qrc"]]))
print(key.select("h", "model_a", "model_b", pl.col("MAE_a").round(4), pl.col("MAE_b").round(4), pl.col("DM_stat").round(2),
                 "DM_p_holm", "better", "significant_5pct"))
rb = C.ramp_breakdown(al, meta["thresholds"])
rb.write_csv(REP / "quantum_ramp_breakdown_test.csv")
print(rb.filter(pl.col("model").is_in(DM_MODELS)).select("h", "model", "n_ramp", pl.col("MAE_normal").round(4),
      pl.col("MAE_ramp").round(3), pl.col("skill_normal").round(3), pl.col("skill_ramp").round(3), pl.col("ramp_F1").round(3)))
wk = pl.concat([C.weekly_skill_mod5(al, [HYB["hqnn"], HYB["ctrl_mlp"], QRCM["qrc"], "gru_uni"] if "gru_uni" in REFS
                                    else [HYB["hqnn"], HYB["ctrl_mlp"], QRCM["qrc"]], h=h).with_columns(pl.lit(h).alias("h")) for h in HORIZONS])
wk.write_csv(REP / "quantum_weekly_skill_test.csv")
print(wk.filter(pl.col("targets") == "all").with_columns(pl.col("skill_MAE").round(3), (pl.col("model") + " h" + pl.col("h").cast(pl.Utf8)).alias("col"))
        .pivot(on="col", index="block_start", values="skill_MAE").sort("block_start"))

shape: (57, 9)
┌─────┬─────────────────────────┬─────────────────────────┬────────┬───┬─────────┬────────────┬─────────────────────────┬──────────────────┐
│ h   ┆ model_a                 ┆ model_b                 ┆ MAE_a  ┆ … ┆ DM_stat ┆ DM_p_holm  ┆ better                  ┆ significant_5pct │
│ --- ┆ ---                     ┆ ---                     ┆ ---    ┆   ┆ ---     ┆ ---        ┆ ---                     ┆ ---              │
│ i64 ┆ str                     ┆ str                     ┆ f64    ┆   ┆ f64     ┆ f64        ┆ str                     ┆ bool             │
╞═════╪═════════════════════════╪═════════════════════════╪════════╪═══╪═════════╪════════════╪═════════════════════════╪══════════════════╡
│ 1   ┆ hqnn_noreup_q4_l2       ┆ ctrl_mlp_q4_l2          ┆ 0.2371 ┆ … ┆ -6.65   ┆ 9.6199e-10 ┆ hqnn_noreup_q4_l2       ┆ true             │
│ 1   ┆ hqnn_noreup_q4_l2       ┆ hqnn_noent_noreup_q4_l2 ┆ 0.2371 ┆ … ┆ -3.77   ┆ 0.002783   ┆ hqnn_noreup_q4_l2       ┆ true             

In [7]:
# Cell 7: summary
summ = {"freeze_record": FREEZE_F.name, "test_predictions_sha256": TEST_SHA,
        "test": BT.select("h", "model", "n_seeds", "MAE_mean", "MAE_sd", "skill_MAE").to_dicts(),
        "dm_key_pairs": key.to_dicts()}
(REP / "quantum_test_summary.json").write_text(json.dumps(summ, indent=1, default=str))
for h in HORIZONS:
    s = ALL.filter(pl.col("h") == h)
    print(f"h={h:>2}: " + " | ".join(f"{m} rank {s.filter(pl.col('model') == m)['rank'][0]} skill {s.filter(pl.col('model') == m)['skill_MAE'][0]:+.3f}"
                                      for m in (HYB["hqnn"], QRCM["qrc"], "gru_uni", "patchtst", "lightgbm") if s.filter(pl.col('model') == m).height))

h= 1: hqnn_noreup_q4_l2 rank 13 skill +0.020 | qrc_q8_d4 rank 17 skill +0.016 | gru_uni rank 8 skill +0.036 | patchtst rank 6 skill +0.048 | lightgbm rank 2 skill +0.064
h= 5: hqnn_noreup_q4_l2 rank 9 skill +0.034 | qrc_q8_d4 rank 18 skill +0.022 | gru_uni rank 5 skill +0.041 | patchtst rank 6 skill +0.039 | lightgbm rank 2 skill +0.046
h=15: hqnn_noreup_q4_l2 rank 6 skill +0.029 | qrc_q8_d4 rank 16 skill +0.015 | gru_uni rank 5 skill +0.031 | patchtst rank 7 skill +0.029 | lightgbm rank 3 skill +0.033


In [9]:
# Cell 8: export outputs (save the notebook first with Cmd+S, then run this cell)
%run ../src/tools/nb_outputs_to_md.py 23_week6_quantum_test.ipynb

written 23_week6_quantum_test_outputs.md | code cells 10 | figures 0
